In [1]:
import numpy as np
import pandas as pd
TARGETS = ["X"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [2]:
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        for col in [col_r2, col_mse]:
            if col in results.columns:
                results[col] = (
                    results[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )

In [5]:
best_models_tables = {}
best_only = []

summary_all = []     # estatísticas com todos
summary_topN = []   # estatísticas com top 10

N = 10

import pandas as pd

def summarize_models(results, TARGETS, SETS, N=10, output="Resumo_Estatisticas.xlsx"):
    
    best_models_tables = {}
    best_only = []
    summary_all = []
    summary_top10 = []

    for target in TARGETS:
        for dataset in SETS:

            col_r2 = f"R2_{dataset}_{target}"
            col_mse = f"MSE_{dataset}_{target}"

            if col_r2 not in results.columns:
                continue

            table = results.copy()

            table = table.sort_values(by=col_r2, ascending=False)

            best_models_tables[f"{dataset}_{target}"] = table

            # 🔹 melhor modelo
            best = table.iloc[0]

            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": best["model"],
                "Neurons": best["Neurons"],
                # "r": best["r"],
                "Best_R2": best[col_r2]
            })

            # 🔹 estatísticas de todos
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })

            # 🔹 estatísticas top N
            topN = table.head(N)

            summary_topN.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": topN[col_r2].mean(),
                "std_r2": topN[col_r2].std(),
                "mean_mse": topN[col_mse].mean(),
                "std_mse": topN[col_mse].std()
            })

    df_summary_all = pd.DataFrame(summary_all)
    df_summary_topN = pd.DataFrame(summary_topN)
    best_only_df = pd.DataFrame(best_only)
    # exportar
    with pd.ExcelWriter(output) as writer:
        df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
        df_summary_topN.to_excel(writer, sheet_name="Top_Modelos", index=False)
        best_only_df.to_excel(writer, sheet_name="Melhor_Modelo", index=False)

    return best_models_tables, df_summary_all, df_summary_topN, best_only_df

In [6]:
tables, summary_all, summary_top10, best_models = summarize_models(
    results, TARGETS, SETS
)

In [8]:
summary_top10["bestR2"] = best_models["Best_R2"]
summary_top10["Neurons"] = best_models["Neurons"]
display(summary_top10)

,dataset,target,mean_r2,std_r2,mean_mse,std_mse,bestR2,Neurons
0,Train_1,X,0.967700,0.007027,0.003483,0.000758,0.979844,"[8, 4]"
1,Train_2,X,0.980699,0.012982,0.001626,0.001094,0.996301,[32]
2,Val,X,0.814659,0.042710,0.017723,0.004084,0.907652,"[32, 16]"
3,Test_1,X,0.975575,0.006997,0.002559,0.000733,0.985139,"[8, 4]"
4,Test_2,X,0.983590,0.009401,0.001350,0.000774,0.997160,[32]
5,Test_3,X,-0.405961,0.339068,0.129355,0.031196,0.337397,"[32, 16]"


- Entradas: SWdRef, SWeRef, Theta
- Saidas: X
- Trajetorias: ZZ + ZZRoted